# ntl-tpu-ll8q14

LoRA24 members (Lucas Boesen's recipe with Dhruv Tibarewal's two stages) trained on the TPU v5e-8 with torch_xla SPMD, one after another: stage 1 on crtproduction's open pool, stage 2 on the official rows per held-out fold and on all rows. Training kernel, not a submission.

In [ ]:
%%writefile /kaggle/working/l24_worker.py
"""Two-stage LoRA24 member trainer for Name That LLM, saved in the format nlm-v27's infer.py loads.

Lucas Boesen's public LoRA24 recipe (kernel name-that-llm-train-lora24-s7) with Dhruv Tibarewal's two stages (nlm-v27:
stage 1 on his 120k public pool with class weights, stage 2 on the 2,699 official rows): the base truncated to --keep
layers in float16 (frozen), LoRA r16 / alpha 32 / dropout .05 on every linear layer of the decoder (fp32), masked mean
pool (first token dropped) of 8 depths with one LayerNorm each, concatenated, into a plain and a prompt-centred linear
head; micro-batches of same-prompt triplets; loss CE(plain(z)) + CE(centred(z - triplet mean)); AdamW (adapter and
head learning rates), weight decay .01, warmup then cosine, clip 1.0, fp16 autocast with a GradScaler, gradient
checkpointing. A multimodal checkpoint (Ministral 3) keeps its text path only; infer.py loads the same AutoModel.

    python l24_worker.py --stage 0 ...                         fp16 health, memory and speed check (one GPU)
    torchrun --nproc_per_node 2 l24_worker.py --stage 1 ...    pool stage, data-parallel over both GPUs, time-boxed
    python l24_worker.py --stage 2 --jobs 0,2,4 ...            official rows, one job per held-out fold (or 'full')

Output of stage 1 and of each stage-2 job: adapter_config.json + adapter_model.safetensors (peft), heads.pt (norms.i,
plain, centered; rows in --labels order) and meta.json; stage-2 fold jobs also write valid.npz (held-out log-probs).
"""
import argparse
import contextlib
import importlib.util as _iu
import json
import math
import os
import time

_fs = _iu.find_spec   # Kaggle ships a torchao that some peft versions reject; it is unused here
_iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _fs(name, *a, **k)
import numpy as np
import pandas as pd
import torch
import torch.distributed
import torch.nn.functional as F
from torch import nn

T0 = time.time()
TARGET = r'(.*\.)?layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|gate_up_proj|down_proj))'


def parse():
    p = argparse.ArgumentParser()
    p.add_argument('--stage', type=int, required=True)
    p.add_argument('--base', required=True); p.add_argument('--out', required=True)
    p.add_argument('--labels', required=True, help='json list: head row order')
    p.add_argument('--keep', type=int, default=24); p.add_argument('--layers', default='2,4,6,8,12,16,20,24')
    p.add_argument('--max-len', type=int, default=384)
    p.add_argument('--r', type=int, default=16); p.add_argument('--alpha', type=int, default=32)
    p.add_argument('--ldrop', type=float, default=0.05)
    p.add_argument('--lr', type=float, default=1e-4); p.add_argument('--head-lr', type=float, default=1e-3)
    p.add_argument('--wd', type=float, default=0.01); p.add_argument('--warmup', type=float, default=0.03)
    p.add_argument('--epochs', type=float, default=2.0)
    p.add_argument('--tri', type=int, default=2, help='triplets per micro-batch')
    p.add_argument('--accum', type=int, default=2, help='micro-batches per optimizer step and GPU')
    p.add_argument('--hours', type=float, default=9.0, help='stage 1: training time budget')
    p.add_argument('--ckpt-min', type=float, default=60.0)
    p.add_argument('--pool', default=''); p.add_argument('--train', default='')
    p.add_argument('--init', default='', help='stage 1: an earlier stage-1 output to continue; stage 2: the stage-1 output')
    p.add_argument('--jobs', default='', help="stage 2: comma list of held-out folds and/or 'full'")
    p.add_argument('--class-weights', action='store_true')
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--check-rows', type=int, default=48, help='stage 0: rows per check')
    a = p.parse_args()
    a.layers = [int(x) for x in a.layers.split(',')]; a.labels = json.loads(a.labels)
    return a


RANK, WORLD = int(os.environ.get('RANK', 0)), int(os.environ.get('WORLD_SIZE', 1))
LOCAL = int(os.environ.get('LOCAL_RANK', 0))
DEV = torch.device(f'cuda:{LOCAL}') if torch.cuda.is_available() else torch.device('cpu')
CUDA = DEV.type == 'cuda'


def log(*x):
    if RANK == 0:
        print(f'[{(time.time() - T0) / 60:7.1f} min]', *x, flush=True)


def autocast():
    return torch.autocast('cuda', dtype=torch.float16) if CUDA else contextlib.nullcontext()


def load_base(a):
    from transformers import AutoConfig, AutoModel, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(a.base)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(a.base); tc = cfg.text_config if hasattr(cfg, 'text_config') else cfg
    tc.num_hidden_layers = a.keep; tc.use_cache = False
    if getattr(tc, 'layer_types', None) is not None:
        tc.layer_types = tc.layer_types[:a.keep]
    m = AutoModel.from_pretrained(a.base, config=cfg, dtype=torch.float16 if CUDA else torch.float32)
    for name in ('vision_tower', 'multi_modal_projector'):   # text only: the image path never runs
        if getattr(m, name, None) is not None:
            setattr(m, name, None)
    m.to(DEV)
    m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    m.enable_input_require_grads()
    return tok, m, tc.hidden_size


def add_lora(m, a, init=''):
    from peft import LoraConfig, PeftModel, get_peft_model
    if init:
        model = PeftModel.from_pretrained(m, init, is_trainable=True)
    else:
        model = get_peft_model(m, LoraConfig(r=a.r, lora_alpha=a.alpha, lora_dropout=a.ldrop, bias='none',
                                             target_modules=TARGET))
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert n > 0, 'no LoRA parameters: target modules did not match'
    return model, n


class Net(nn.Module):
    """LoRA decoder + one LayerNorm per pooled depth + plain and centred heads; forward returns the loss."""

    def __init__(self, model, hid, layers, k):
        super().__init__()
        self.model, self.layers = model, layers
        self.norms = nn.ModuleList([nn.LayerNorm(hid) for _ in layers])
        self.plain = nn.Linear(hid * len(layers), k)
        self.centered = nn.Linear(hid * len(layers), k)
        self.drop = nn.Dropout(0.1)

    def features(self, ids, att):
        with autocast():
            hs = self.model(input_ids=ids, attention_mask=att, output_hidden_states=True, use_cache=False).hidden_states
        m = att.unsqueeze(-1).float()
        m[:, 0] = 0
        return torch.cat([nm((hs[l].float() * m).sum(1) / m.sum(1)) for l, nm in zip(self.layers, self.norms)], -1)

    def forward(self, ids, att, y, same, cw):
        z = self.features(ids, att)
        zt = z.view(z.shape[0] // 3, 3, -1)
        zc = (zt - zt.mean(1, keepdim=True)).reshape(z.shape[0], -1)
        return wce(self.plain(self.drop(z)), y, cw) + wce(self.centered(self.drop(zc)), y, cw, same.repeat_interleave(3))


def wce(logits, y, cw, mask=None):
    nll = F.cross_entropy(logits.float(), y, reduction='none')
    w = cw[y] if cw is not None else torch.ones_like(nll)
    if mask is not None:
        w = w * mask
    return (nll * w).sum() / w.sum().clamp(min=1e-6)


def encode(tok, texts, max_len):
    """infer.py's tokenisation: default special tokens, truncation at max_len, no end token."""
    return tok(list(texts), truncation=True, max_length=max_len)['input_ids']


def batch(enc, idx, pad):
    Lm = max(len(enc[i]) for i in idx)
    ids = torch.full((len(idx), Lm), pad, dtype=torch.long); att = torch.zeros((len(idx), Lm), dtype=torch.long)
    for r, i in enumerate(idx):
        ids[r, :len(enc[i])] = torch.tensor(enc[i]); att[r, :len(enc[i])] = 1
    return ids.to(DEV), att.to(DEV)


def triplets(groups, rows, rng):
    """Same-group triplets over `rows` (a short last chunk is filled with other members of its group); rows whose
    group has one member go into mixed triplets, flagged 0 so their centred loss is skipped."""
    by = {}
    for r in rows:
        by.setdefault(groups[r], []).append(int(r))
    out, solo = [], []
    for g in sorted(by):
        m = [int(x) for x in rng.permutation(by[g])]
        if len(m) == 1:
            solo.append(m[0]); continue
        for s in range(0, len(m), 3):
            b = m[s:s + 3]
            while len(b) < 3:
                b.append(int(rng.choice([x for x in m if x not in b] or m)))
            out.append((b, 1.0))
    solo = [int(x) for x in rng.permutation(solo)]
    out += [(solo[s:s + 3], 0.0) for s in range(0, len(solo) - 2, 3)]
    return [out[i] for i in rng.permutation(len(out))]


def micro_batches(trips, enc, tri, rng, window=64):
    """`tri` triplets per micro-batch, length-sorted within windows of the shuffled list, then shuffled again."""
    mbs = []
    for s in range(0, len(trips), window):
        w = sorted(trips[s:s + window], key=lambda t: max(len(enc[i]) for i in t[0]))
        mbs += [w[i:i + tri] for i in range(0, len(w) - tri + 1, tri)]
    return [mbs[i] for i in rng.permutation(len(mbs))]


def lr_mult(step, total, warm):
    if step < warm:
        return (step + 1) / warm
    return 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warm) / max(1, total - warm))))


def optimizer(net, a):
    ad = [p for p in net.model.parameters() if p.requires_grad]
    hd = list(net.norms.parameters()) + list(net.plain.parameters()) + list(net.centered.parameters())
    opt = torch.optim.AdamW([{'params': ad, 'lr': a.lr}, {'params': hd, 'lr': a.head_lr}], weight_decay=a.wd)
    return opt, ad + hd


def checkpointed(model):
    """Decoder layers that will run gradient checkpointing (the flag is set and the layer is in training mode)."""
    n = sum(1 for m_ in model.modules() if getattr(m_, 'gradient_checkpointing', False) is True and m_.training)
    assert n > 0, 'gradient checkpointing is off: the decoder would keep every activation'
    return n


def zero_loss(params):
    """A loss of exactly zero that reaches every trainable parameter: stands in for a micro-batch whose fp16 forward
    overflowed, so every rank still runs the same backward (and gradient all-reduce)."""
    return sum(p.sum() for p in params) * 0.0


@torch.no_grad()
def predict(net, enc, rows, pad, bs=16):
    net.eval()
    out = np.zeros((len(rows), net.plain.out_features), np.float32)
    order = np.argsort([len(enc[i]) for i in rows], kind='stable')
    for s in range(0, len(order), bs):
        pos = order[s:s + bs]
        ids, att = batch(enc, [rows[i] for i in pos], pad)
        out[pos] = F.log_softmax(net.plain(net.features(ids, att)), -1).float().cpu().numpy()
    net.train()
    return out


def save(peft_model, net, out, meta):
    os.makedirs(out, exist_ok=True)
    peft_model.save_pretrained(out)
    H = {}
    for i, nm in enumerate(net.norms):
        H[f'norms.{i}.weight'] = nm.weight.detach().float().cpu().clone(); H[f'norms.{i}.bias'] = nm.bias.detach().float().cpu().clone()
    for h in ('plain', 'centered'):
        H[f'{h}.weight'] = getattr(net, h).weight.detach().float().cpu().clone()
        H[f'{h}.bias'] = getattr(net, h).bias.detach().float().cpu().clone()
    torch.save(H, os.path.join(out, 'heads.pt'))
    json.dump(meta, open(os.path.join(out, 'meta.json'), 'w'), indent=1)


def load_heads(net, d):
    H = torch.load(os.path.join(d, 'heads.pt'), map_location='cpu')
    with torch.no_grad():
        for i, nm in enumerate(net.norms):
            nm.weight.copy_(H[f'norms.{i}.weight']); nm.bias.copy_(H[f'norms.{i}.bias'])
        for h in ('plain', 'centered'):
            getattr(net, h).weight.copy_(H[f'{h}.weight']); getattr(net, h).bias.copy_(H[f'{h}.bias'])


def step_fn(net, wrapped, opt, scaler, params, a):
    """One micro-batch: forward (loss), backward; returns (loss value, overflowed)."""
    def run(mb, enc, y_all, pad, cw, sync):
        idx = [i for t, _ in mb for i in t]
        ids, att = batch(enc, idx, pad)
        y = torch.as_tensor(y_all[idx], device=DEV)
        same = torch.tensor([s for _, s in mb], device=DEV, dtype=torch.float32)
        ctx = wrapped.no_sync() if (WORLD > 1 and not sync) else contextlib.nullcontext()
        with ctx:
            loss = wrapped(ids, att, y, same, cw)
            bad = not bool(torch.isfinite(loss))
            if bad:
                loss = zero_loss(params)
            scaler.scale(loss / a.accum).backward()
        return float(loss.detach()), bad
    return run


def stage0(a):
    """fp16 health (non-finite pooled depths on the longest pool texts), peak memory of a full-length training step,
    and training speed on random triplets. Writes <out>/check.json."""
    pool = pd.read_parquet(a.pool)
    tok, m, hid = load_base(a)
    model, n_lora = add_lora(m, a)
    net = Net(model, hid, a.layers, len(a.labels)).to(DEV)
    enc = encode(tok, pool.text.astype(str).tolist(), a.max_len)
    lens = np.array([len(x) for x in enc])
    longest = [int(i) for i in np.argsort(-lens)[:a.check_rows]]
    bad_rows = 0
    with torch.no_grad():
        net.eval()
        for s in range(0, len(longest), 6):
            ids, att = batch(enc, longest[s:s + 6], tok.pad_token_id)
            with autocast():
                hs = model(input_ids=ids, attention_mask=att, output_hidden_states=True, use_cache=False).hidden_states
            fin = torch.stack([torch.isfinite(hs[l]).flatten(1).all(1) for l in a.layers]).all(0)
            bad_rows += int((~fin).sum())
        net.train()
    log(f'fp16 check: {bad_rows}/{len(longest)} of the longest pool texts overflow at a pooled depth; '
        f'checkpointed layers {checkpointed(model)}')
    y_all = pool.model.map({l: i for i, l in enumerate(a.labels)}).fillna(0).astype(int).values
    opt, params = optimizer(net, a)
    scaler = torch.amp.GradScaler('cuda', init_scale=256.0, enabled=CUDA)
    run = step_fn(net, net, opt, scaler, params, a)
    if CUDA:
        torch.cuda.reset_peak_memory_stats(DEV)
    t = time.time()
    trips = [([longest[3 * k], longest[3 * k + 1], longest[3 * k + 2]], 1.0) for k in range(4)]
    tri = a.tri
    while True:   # full-length micro-batches: the memory high-water mark; halve the micro-batch on running out
        try:
            for k in range(2):
                run(trips[2 * k:2 * k + tri], enc, y_all, tok.pad_token_id, None, True)
            break
        except torch.cuda.OutOfMemoryError:
            opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
            assert tri > 1, 'out of GPU memory at one triplet per micro-batch'
            log(f'out of memory at {tri} triplets per micro-batch; trying {tri // 2}'); tri //= 2
    a.tri = tri
    peak = torch.cuda.max_memory_allocated(DEV) / 2 ** 30 if CUDA else 0.0
    opt.zero_grad(set_to_none=True)
    rng = np.random.RandomState(a.seed)
    groups = pool.prompt_key.astype(str).values
    mbs = micro_batches(triplets(groups, rng.choice(len(pool), min(3000, len(pool)), replace=False), rng), enc, a.tri, rng)[:12]
    if CUDA:
        torch.cuda.synchronize()
    t = time.time(); toks = 0; bad = 0
    for mb in mbs:
        _, b = run(mb, enc, y_all, tok.pad_token_id, None, True); bad += b
        toks += sum(len(enc[i]) for t_, _ in mb for i in t_)
    if CUDA:
        torch.cuda.synchronize()
    tps = toks / (time.time() - t)
    total_tokens = int(lens.sum())
    res = dict(ok=bad_rows <= 0.02 * len(longest) and bad == 0, overflow_rows=bad_rows, rows_checked=len(longest),
               peak_gib=round(peak, 2), tokens_per_s_one_gpu=round(tps, 1), lora_params=n_lora, pool_tokens=total_tokens,
               hours_per_epoch_two_gpus=round(total_tokens / (2 * tps) / 3600, 2), train_step_overflow=bad, hidden=hid,
               tri=tri)
    log('check', json.dumps(res))
    os.makedirs(a.out, exist_ok=True)
    json.dump(res, open(os.path.join(a.out, 'check.json'), 'w'), indent=1)


def stage1(a):
    if CUDA:
        torch.cuda.set_device(DEV)
    torch.manual_seed(a.seed); np.random.seed(a.seed)
    pool = pd.read_parquet(a.pool)
    y_all = pool.model.map({l: i for i, l in enumerate(a.labels)}).values
    assert not np.isnan(y_all.astype(float)).any(), 'pool label outside --labels'
    y_all = y_all.astype(int)
    tok, m, hid = load_base(a)
    model, n_lora = add_lora(m, a, a.init)   # before the process group: once one exists, peft loads an adapter on its tensor-parallel path
    if WORLD > 1:
        torch.distributed.init_process_group('nccl' if CUDA else 'gloo')
    net = Net(model, hid, a.layers, len(a.labels)).to(DEV)
    prior = 0.0
    if a.init:   # continue an earlier stage-1 run: its adapter and heads, a new triplet order (--seed), a new cosine cycle
        load_heads(net, a.init)
        prev = json.load(open(os.path.join(a.init, 'meta.json')))
        prior = prev.get('epochs_total', prev['epochs_done'])
    net.train()   # from_pretrained returns eval mode, in which the decoder layers skip gradient checkpointing
    n_ckpt = checkpointed(model)
    wrapped = net
    if WORLD > 1:
        wrapped = nn.parallel.DistributedDataParallel(net, device_ids=[LOCAL] if CUDA else None,
                                                      gradient_as_bucket_view=True)
    enc = encode(tok, pool.text.astype(str).tolist(), a.max_len)
    counts = np.bincount(y_all, minlength=len(a.labels)).astype(float)
    cw = torch.tensor(len(y_all) / (len(a.labels) * np.maximum(counts, 1)), dtype=torch.float32, device=DEV) \
        if a.class_weights else None
    groups = pool.prompt_key.astype(str).values
    rng = np.random.RandomState(a.seed)
    mbs = []
    for e in range(int(math.ceil(a.epochs))):
        ep = micro_batches(triplets(groups, np.arange(len(pool)), rng), enc, a.tri, rng)
        if e + 1 > a.epochs:
            ep = ep[:int(len(ep) * (a.epochs - e))]
        mbs += ep
    mine = mbs[RANK::WORLD][:len(mbs) // WORLD]
    planned = len(mine) // a.accum
    log(f'stage 1: {len(pool)} pool rows, {len(mbs)} micro-batches of {a.tri} triplets over {a.epochs} epochs, '
        f'{planned} optimizer steps per rank ({WORLD} ranks x {a.accum} accumulation), LoRA params {n_lora}, '
        f'class weights {"on" if cw is not None else "off"}, checkpointed layers {n_ckpt}')
    opt, params = optimizer(net, a)
    scaler = torch.amp.GradScaler('cuda', init_scale=256.0, enabled=CUDA)
    run = step_fn(net, wrapped, opt, scaler, params, a)
    total, warm = planned, max(1, int(a.warmup * planned))
    budget = a.hours * 3600; t_train = time.time(); last_ckpt = time.time()
    step, losses, bad, skipped, t_cal = 0, [], 0, 0, None
    meta = dict(stage=1, base=a.base, keep=a.keep, layers=a.layers, max_len=a.max_len, labels=a.labels, r=a.r,
                alpha=a.alpha, lr=a.lr, head_lr=a.head_lr, epochs=a.epochs, tri=a.tri, accum=a.accum, world=WORLD,
                class_weights=cw is not None, pool_rows=len(pool), init=a.init, seed=a.seed)
    for i, mb in enumerate(mine):
        sync = (i + 1) % a.accum == 0
        lv, b = run(mb, enc, y_all, tok.pad_token_id, cw, sync)
        losses.append(lv); bad += b
        if not sync:
            continue
        for g, base_lr in zip(opt.param_groups, (a.lr, a.head_lr)):
            g['lr'] = base_lr * lr_mult(step, total, warm)
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        before = scaler.get_scale(); scaler.step(opt); scaler.update()
        skipped += scaler.get_scale() < before
        opt.zero_grad(set_to_none=True); step += 1
        if step == 10:
            t_cal = time.time()
        if step == 60 or (step > 60 and step % 200 == 0):
            # the time box: steps that fit the remaining budget at the measured speed (rank 0 decides for all)
            sps = (time.time() - t_cal) / (step - 10)
            fit = step + int((budget - (time.time() - t_train)) / sps)
            new_total = torch.tensor([min(planned, max(step + 1, fit))], device=DEV)
            if WORLD > 1:
                torch.distributed.broadcast(new_total, 0)
            if step == 60:
                total = int(new_total.item()); warm = min(warm, max(1, total // 20))
                log(f'speed {sps:.2f} s/step: {total} of {planned} planned steps fit the {a.hours} h budget '
                    f'({total / planned * a.epochs:.2f} epochs)')
            elif int(new_total.item()) < total:
                total = int(new_total.item()); log(f'running slower: total cut to {total} steps')
        if step % 100 == 0:
            log(f'step {step}/{total} loss {np.mean(losses[-100 * a.accum:]):.4f} overflow {bad} skipped {skipped} '
                f'lr {opt.param_groups[0]["lr"]:.2e} ({(time.time() - t_train) / 3600:.2f} h)')
        if RANK == 0 and time.time() - last_ckpt > a.ckpt_min * 60:
            save(model, net, os.path.join(a.out, 'ckpt'), dict(meta, step=step, total=total)); last_ckpt = time.time()
        if step >= total:
            break
    hours = (time.time() - t_train) / 3600
    meta.update(steps=step, total_steps=total, planned_steps=planned, hours=round(hours, 3), overflow_microbatches=bad,
                skipped_steps=int(skipped), final_loss=float(np.mean(losses[-100 * a.accum:])),
                epochs_done=round(step / planned * a.epochs, 3))
    meta['epochs_total'] = round(prior + meta['epochs_done'], 3)   # pool epochs including the continued run's
    if RANK == 0:
        save(model, net, a.out, meta)
        log(f'stage 1 done: {step} steps in {hours:.2f} h, {meta["epochs_done"]} epochs, overflow micro-batches {bad}, '
            f'skipped steps {skipped}; saved {a.out}')
    if WORLD > 1:
        torch.distributed.barrier(); torch.distributed.destroy_process_group()


def stage2(a):
    torch.manual_seed(a.seed)
    os.makedirs(os.path.dirname(a.out.rstrip('/')) or '.', exist_ok=True)
    tr = pd.read_parquet(a.train)
    y_all = tr.label.astype(int).values; folds = tr.fold.astype(int).values
    tok, m, hid = load_base(a)
    model, n_lora = add_lora(m, a, a.init)
    net = Net(model, hid, a.layers, len(a.labels)).to(DEV)
    from peft import get_peft_model_state_dict, set_peft_model_state_dict
    s1 = {k: v.detach().clone() for k, v in get_peft_model_state_dict(model).items()}
    enc = encode(tok, tr.text.astype(str).tolist(), a.max_len)
    groups = tr.prompt_id.astype(str).values
    s1_meta = json.load(open(os.path.join(a.init, 'meta.json')))
    if '0' in a.jobs.split(','):   # the pool-only model on the official rows, none of which it has seen
        lp = predict(net, enc, list(range(len(tr))), tok.pad_token_id)
        s1_meta['official_balanced_accuracy'] = float(np.mean([(lp[y_all == c].argmax(1) == c).mean() for c in np.unique(y_all)]))
        np.save(a.out + '_stage1_on_train.npy', lp)
        log(f"stage-1 model on the official rows: balanced accuracy {s1_meta['official_balanced_accuracy']:.4f}")
    for job in a.jobs.split(','):
        t0 = time.time()
        set_peft_model_state_dict(model, s1); load_heads(net, a.init)
        rows = np.arange(len(tr)) if job == 'full' else np.where(folds != int(job))[0]
        rng = np.random.RandomState(a.seed + (99 if job == 'full' else int(job)))
        mbs = []
        for e in range(int(math.ceil(a.epochs))):
            mbs += micro_batches(triplets(groups, rows, rng), enc, a.tri, rng)
        total = len(mbs) // a.accum; warm = max(1, int(a.warmup * total))
        opt, params = optimizer(net, a)
        scaler = torch.amp.GradScaler('cuda', init_scale=256.0, enabled=CUDA)
        run = step_fn(net, net, opt, scaler, params, a)
        net.train(); checkpointed(model)   # see stage1: gradient checkpointing only runs in training mode
        step, losses, bad, skipped = 0, [], 0, 0
        for i, mb in enumerate(mbs[:total * a.accum]):
            lv, b = run(mb, enc, y_all, tok.pad_token_id, None, True)
            losses.append(lv); bad += b
            if (i + 1) % a.accum:
                continue
            for g, base_lr in zip(opt.param_groups, (a.lr, a.head_lr)):
                g['lr'] = base_lr * lr_mult(step, total, warm)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            before = scaler.get_scale(); scaler.step(opt); scaler.update()
            skipped += scaler.get_scale() < before
            opt.zero_grad(set_to_none=True); step += 1
            if step % 50 == 0:
                log(f'job {job} step {step}/{total} loss {np.mean(losses[-50 * a.accum:]):.4f} overflow {bad} skipped {skipped}')
        name = os.path.basename(a.out.rstrip('/')) + ('' if job == 'full' else f'_f{job}')
        out = os.path.join(os.path.dirname(a.out.rstrip('/')), name)
        check = predict(net, enc, list(range(40)), tok.pad_token_id)
        meta = dict(s1_meta, stage=2, job=job, steps=step, epochs=a.epochs, lr=a.lr, head_lr=a.head_lr, tri=a.tri,
                    accum=a.accum, overflow_microbatches=bad, skipped_steps=int(skipped), init=a.init,
                    final_loss=float(np.mean(losses[-50 * a.accum:])), minutes=round((time.time() - t0) / 60, 2),
                    in_sample_check_40=float((check.argmax(1) == y_all[:40]).mean()))
        if job != 'full':
            valid = np.where(folds == int(job))[0]
            lp = predict(net, enc, list(valid), tok.pad_token_id)
            meta['valid_accuracy'] = float((lp.argmax(1) == y_all[valid]).mean())
            meta['valid_balanced_accuracy'] = float(np.mean([(lp[y_all[valid] == c].argmax(1) == c).mean()
                                                            for c in np.unique(y_all[valid])]))
        save(model, net, out, meta)
        np.save(os.path.join(out, 'check40.npy'), check)
        if job != 'full':
            np.savez(os.path.join(out, 'valid.npz'), ids=np.array(tr.id.astype(str).values[valid].tolist()), logits=lp)
        log(f"job {job} done: {step} steps, {meta['minutes']} min, in-sample 40 {meta['in_sample_check_40']:.2f}"
            + (f", held-out accuracy {meta['valid_accuracy']:.4f} balanced {meta['valid_balanced_accuracy']:.4f}"
               if job != 'full' else '') + f'; saved {out}')


if __name__ == '__main__':
    args = parse()
    {0: stage0, 1: stage1, 2: stage2}[args.stage](args)

In [ ]:
%%writefile /kaggle/working/tpu_member.py
"""A LoRA24 member trained end to end on a Kaggle TPU v5e-8: stage 1 on a public pool, then stage 2 on the official
rows once per held-out fold and once on all of them, with out-of-fold log-probs for the replica.

l24_worker.py's recipe (base truncated to --keep layers and frozen, LoRA on every decoder projection, masked mean pool
of 8 depths with one LayerNorm each into a plain and a prompt-centred head, same-prompt triplets, class-weighted
cross-entropy, AdamW with warmup and cosine, clip 1.0) and its output layout, which infer.py's l24_predict loads as is.
What the TPU changes (after GoPain's forum measurements, 2026-11-01):
- one process, data parallel over every chip with torch_xla SPMD (the batch dimension sharded, the model replicated);
- bf16 base (no fp16 overflow, no GradScaler), fp32 LoRA and heads;
- fixed batch shapes: a triplet goes to the smallest (tokens, rows) bucket that holds its longest passage, every
  bucket spends about the same tokens per step, and leftovers that fill no whole batch are dropped, so each shape
  compiles once;
- torch_xla's own gradient checkpointing on every decoder layer (torch's is undone by XLA's common-subexpression pass);
- LoRA dropout 0 (the recomputed forward would need the dropout mask replayed).

    PJRT_DEVICE=TPU python tpu_member.py --base DIR --pool pool.parquet --train train.parquet --labels JSON --out DIR
    python tpu_member.py --cpu ...        the same on the CPU in float32, without torch_xla (local smoke test)

pool.parquet: text, model, prompt_key. train.parquet: id, text, label (index into --labels), fold, prompt_id.
Writes <out>_s1/ (stage 1), <out>_f<k>/ and <out>/ (stage 2), s1_on_train.npy, oof.npz (ids, labels, logits: the
held-out fold predictions of the fold jobs, every official row once) and summary.json.
"""
import argparse
import functools
import json
import math
import os
import sys
import time

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import l24_worker as W   # noqa: E402

T0 = time.time()
S1_SHAPES = [(96, 216), (160, 144), (256, 96), (384, 48)]   # (tokens, rows); rows are multiples of 3 x 8 chips
S2_SHAPES = [(96, 72), (160, 48), (256, 24), (384, 24)]
EVAL_SHAPES = [(96, 128), (160, 96), (256, 64), (384, 32)]


def log(*x):
    print(f'[{(time.time() - T0) / 60:7.1f} min]', *x, flush=True)


def parse():
    p = argparse.ArgumentParser()
    p.add_argument('--cpu', action='store_true')
    p.add_argument('--base', required=True); p.add_argument('--out', required=True)
    p.add_argument('--labels', required=True); p.add_argument('--pool', required=True); p.add_argument('--train', required=True)
    p.add_argument('--keep', type=int, default=24); p.add_argument('--layers', default='2,4,6,8,12,16,20,24')
    p.add_argument('--max-len', type=int, default=384)
    p.add_argument('--r', type=int, default=16); p.add_argument('--alpha', type=int, default=32)
    p.add_argument('--lr', type=float, default=2e-4); p.add_argument('--head-lr', type=float, default=2e-3)
    p.add_argument('--lr2', type=float, default=8e-5); p.add_argument('--head-lr2', type=float, default=8e-4)
    p.add_argument('--wd', type=float, default=0.01)
    p.add_argument('--warmup', type=float, default=0.03); p.add_argument('--warmup2', type=float, default=0.1)
    p.add_argument('--epochs', type=float, default=2.0); p.add_argument('--epochs2', type=float, default=2.0)
    p.add_argument('--hours', type=float, default=3.0, help='stage 1 time box')
    p.add_argument('--jobs', default='0,1,2,3,4,full')
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--tp', type=int, default=1, help='tensor parallel degree (Megatron-style weight sharding over this many '
                   'chips; the rest of the chips split the batch), for bases too large to replicate on one chip')
    p.add_argument('--pool-weight', type=float, default=0.0, help='stage 1: loss weight of each pool row = match_weight '
                   '** this, normalised to mean 1 within each label and clipped to [1/c, c] (0: every row weighs 1)')
    p.add_argument('--pool-weight-clip', type=float, default=4.0)
    p.add_argument('--shapes-scale', type=float, default=1.0, help='CPU smoke test: rows per shape x this (min 3)')
    a = p.parse_args()
    a.layers = [int(x) for x in a.layers.split(',')]; a.labels = json.loads(a.labels)
    a.ldrop = 0.0
    return a


class Runtime:
    """SPMD data parallel on the XLA device, or one CPU process."""

    def __init__(self, cpu, tp=1):
        self.cpu = cpu
        if cpu:
            self.dev, self.n = torch.device('cpu'), 1
            return
        import torch_xla
        import torch_xla.core.xla_model as xm
        import torch_xla.distributed.spmd as xs
        import torch_xla.runtime as xr
        xr.use_spmd()
        self.xm, self.xs = xm, xs
        self.n = xr.global_runtime_device_count()
        assert self.n % tp == 0, (self.n, tp)
        self.dp = self.n // tp
        self.mesh = xs.Mesh(np.arange(self.n), (self.dp, tp), ('data', 'model'))
        self.dev = torch_xla.device() if hasattr(torch_xla, 'device') else xm.xla_device()

    def put(self, t):
        t = t.to(self.dev)
        if not self.cpu:   # batch rows over the chips
            self.xs.mark_sharding(t, self.mesh, ('data',) + (None,) * (t.dim() - 1))
        return t

    def step(self):
        if not self.cpu:
            self.xm.mark_step()


COLUMN = ('q_proj', 'k_proj', 'v_proj', 'gate_proj', 'up_proj')   # output features split over 'model'
ROW = ('o_proj', 'down_proj')                                      # input features split over 'model'


def tp_spec(name, p):
    """Megatron-style split of a decoder projection's base weight or LoRA factor, or None (replicated)."""
    parts = name.split('.')
    proj = next((x for x in parts if x in COLUMN + ROW), None)
    if proj is None or p.dim() != 2:
        return None
    if proj in COLUMN and 'lora_A' not in parts:    # base [out, in] and lora_B [out, r]: split out
        return ('model', None)
    if proj in ROW and 'lora_B' not in parts:       # base [out, in] and lora_A [r, in]: split in
        return (None, 'model')
    return None


def shard_tp(model, rt):
    """Annotates every decoder projection (base weight and LoRA factor on the split side); XLA's SPMD partitioner
    inserts the all-reduces. Heads (q 40 / kv 8 for Qwen3-14B) must divide by the tp degree."""
    n = 0
    for name, p in model.named_parameters():
        spec = tp_spec(name, p)
        if spec is not None:
            rt.xs.mark_sharding(p, rt.mesh, spec); n += 1
    return n


def place(net, rt, tp):
    """Moves the model to the device. With tensor parallelism one parameter at a time, each split right after its
    transfer: an unannotated transfer is replicated on every chip, and Qwen3-14B's 27 layers (19 GB in bf16) do not fit
    one chip's 16 GB."""
    if rt.cpu or tp == 1:
        return net.to(rt.dev), 0
    n = 0
    for name, p in net.named_parameters():
        p.data = p.data.to(rt.dev)
        spec = tp_spec(name, p)
        if spec is not None:
            rt.xs.mark_sharding(p, rt.mesh, spec); n += 1
    for mod in net.modules():
        for k, b in list(mod._buffers.items()):
            if b is not None:
                mod._buffers[k] = b.to(rt.dev)
    return net, n


class Net(W.Net):
    """l24_worker's Net; forward also takes per-row loss weights (stage 1's pool weighting), which multiply the class
    weights in the plain loss and the same-group mask in the centred one."""

    def forward(self, ids, att, y, same, cw, rw=None):
        if rw is None:
            return super().forward(ids, att, y, same, cw)
        z = self.features(ids, att)
        zt = z.view(z.shape[0] // 3, 3, -1)
        zc = (zt - zt.mean(1, keepdim=True)).reshape(z.shape[0], -1)
        return W.wce(self.plain(self.drop(z)), y, cw, rw) + W.wce(self.centered(self.drop(zc)), y, cw, same.repeat_interleave(3) * rw)


def pool_weights(pool, p, c):
    """match_weight ** p, mean 1 within each label, clipped to [1/c, c], mean 1 again (the class balance is kept)."""
    w = np.clip(pool.match_weight.astype(float).fillna(1.0).to_numpy(), 1e-3, None) ** p
    lab = pool.model.to_numpy()
    norm = lambda v: pd.Series(v).groupby(lab).transform(lambda s: s / s.mean()).to_numpy()
    return norm(np.clip(norm(w), 1 / c, c)).astype(np.float32)


def load_base(a, dtype):
    from transformers import AutoConfig, AutoModel, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(a.base)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(a.base); tc = cfg.text_config if hasattr(cfg, 'text_config') else cfg
    tc.num_hidden_layers = a.keep; tc.use_cache = False
    if getattr(tc, 'layer_types', None) is not None:
        tc.layer_types = tc.layer_types[:a.keep]
    try:
        m = AutoModel.from_pretrained(a.base, config=cfg, dtype=dtype)
    except TypeError:   # transformers < 4.56 calls it torch_dtype
        m = AutoModel.from_pretrained(a.base, config=cfg, torch_dtype=dtype)
    m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    m.enable_input_require_grads()
    return tok, m, tc.hidden_size


def xla_checkpointing(model):
    """Swap the checkpoint function of every layer that checkpoints for torch_xla's (optimisation barriers)."""
    import inspect
    from torch_xla.utils.checkpoint import checkpoint
    f = functools.partial(checkpoint, use_reentrant=True) if 'use_reentrant' in inspect.signature(checkpoint).parameters \
        else checkpoint   # older torch_xla: always the reentrant form
    n = 0
    for m in model.modules():
        if getattr(m, 'gradient_checkpointing', False) is True:
            m._gradient_checkpointing_func = f; n += 1
    assert n > 0, 'no layer checkpoints'
    return n


def scaled(shapes, s):
    return [(L, max(3, int(round(R * s / 3)) * 3)) for L, R in shapes]


def buckets(trips, enc, shapes, rng):
    """Micro-batches (tokens, triplets): each triplet in the smallest bucket that holds its longest row."""
    by = {L: [] for L, _ in shapes}
    for t in trips:
        m = max(len(enc[i]) for i in t[0])
        by[next((L for L, _ in shapes if m <= L), shapes[-1][0])].append(t)
    out = []
    for L, R in shapes:
        k = R // 3
        out += [(L, by[L][s:s + k]) for s in range(0, len(by[L]) - k + 1, k)]
    return [out[i] for i in rng.permutation(len(out))]


def tensors(enc, idx, L, pad):
    ids = torch.full((len(idx), L), pad, dtype=torch.long); att = torch.zeros((len(idx), L), dtype=torch.long)
    for r, i in enumerate(idx):
        x = enc[i][:L]
        ids[r, :len(x)] = torch.tensor(x); att[r, :len(x)] = 1
    return ids, att


def train_loop(net, params, batches, enc, y_all, cw, pad, rt, lr, head_lr, warmup, hours=None, tag='', rw_all=None):
    opt = torch.optim.AdamW([{'params': [p for p in net.model.parameters() if p.requires_grad], 'lr': lr},
                             {'params': list(net.norms.parameters()) + list(net.plain.parameters())
                              + list(net.centered.parameters()), 'lr': head_lr}], weight_decay=0.01)
    total = len(batches); warm = max(1, int(warmup * total)); losses = []
    t_train = time.time(); t_cal = None; step = 0
    net.train()
    for L, mb in batches:
        idx = [i for t, _ in mb for i in t]
        ids, att = tensors(enc, idx, L, pad)
        y = rt.put(torch.as_tensor(y_all[idx]))
        same = torch.tensor([s for _, s in mb], dtype=torch.float32).to(rt.dev)
        if rw_all is None:
            loss = net(rt.put(ids), rt.put(att), y, same, cw)
        else:
            loss = net(rt.put(ids), rt.put(att), y, same, cw, rt.put(torch.as_tensor(rw_all[idx])))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        for g, base in zip(opt.param_groups, (lr, head_lr)):
            g['lr'] = base * W.lr_mult(step, total, warm)
        opt.step(); opt.zero_grad(set_to_none=True); rt.step(); step += 1
        losses.append(loss.detach())
        if step == 20:
            t_cal = time.time()
        if hours is not None and step == 120:   # the time box, at the speed measured past the compiles
            sps = (time.time() - t_cal) / 100
            fit = step + int((hours * 3600 - (time.time() - t_train)) / sps)
            if fit < total:
                total = max(step + 1, fit); warm = min(warm, max(1, total // 20))
            log(f'{tag} speed {sps:.2f} s/step: {total} of {len(batches)} steps fit {hours} h')
        if step % 100 == 0 or step >= total:
            log(f'{tag} step {step}/{total} loss {float(torch.stack(losses[-100:]).mean()):.4f} '
                f'lr {opt.param_groups[0]["lr"]:.2e} ({(time.time() - t_train) / 3600:.2f} h)')
        if step >= total:
            break
    del opt
    return dict(steps=step, planned=len(batches), hours=round((time.time() - t_train) / 3600, 3),
                final_loss=float(torch.stack(losses[-100:]).mean()) if losses else float('nan'))


@torch.no_grad()
def predict(net, enc, rows, pad, rt, shapes):
    net.eval()
    out = np.zeros((len(rows), net.plain.out_features), np.float32)
    by = {}
    for p, r in enumerate(rows):
        by.setdefault(next((L for L, _ in shapes if len(enc[r]) <= L), shapes[-1][0]), []).append(p)
    for L, R in shapes:
        ps = by.get(L, [])
        for s in range(0, len(ps), R):
            chunk = ps[s:s + R]
            idx = [rows[p] for p in chunk]
            ids, att = tensors(enc, idx + [idx[0]] * (R - len(idx)), L, pad)
            z = net.plain(net.features(rt.put(ids), rt.put(att)))
            lp = torch.log_softmax(z.float(), -1)
            rt.step()
            out[chunk] = lp.cpu().numpy()[:len(chunk)]
    net.train()
    return out


def save(model, net, out, meta):
    sd = {n: p.detach().cpu().float() for n, p in model.named_parameters() if 'lora_' in n}
    os.makedirs(out, exist_ok=True)
    model.save_pretrained(out, state_dict=sd)
    H = {}
    for i, nm in enumerate(net.norms):
        H[f'norms.{i}.weight'] = nm.weight.detach().float().cpu().clone(); H[f'norms.{i}.bias'] = nm.bias.detach().float().cpu().clone()
    for h in ('plain', 'centered'):
        H[f'{h}.weight'] = getattr(net, h).weight.detach().float().cpu().clone()
        H[f'{h}.bias'] = getattr(net, h).bias.detach().float().cpu().clone()
    torch.save(H, os.path.join(out, 'heads.pt'))
    json.dump(meta, open(os.path.join(out, 'meta.json'), 'w'), indent=1)


def bacc(lp, y):
    return float(np.mean([(lp[y == c].argmax(1) == c).mean() for c in np.unique(y)]))


def main():
    a = parse()
    rt = Runtime(a.cpu, a.tp)
    torch.manual_seed(a.seed); np.random.seed(a.seed)
    s1_shapes, s2_shapes, ev_shapes = (S1_SHAPES, S2_SHAPES, EVAL_SHAPES) if a.shapes_scale == 1 else \
        (scaled(S1_SHAPES, a.shapes_scale), scaled(S2_SHAPES, a.shapes_scale), scaled(EVAL_SHAPES, a.shapes_scale))
    if not a.cpu:
        assert all(R % (3 * rt.dp) == 0 for _, R in s1_shapes + s2_shapes) and all(R % rt.dp == 0 for _, R in ev_shapes)
    pool = pd.read_parquet(a.pool); tr = pd.read_parquet(a.train)
    L2I = {l: i for i, l in enumerate(a.labels)}
    yp = pool.model.map(L2I); assert yp.notna().all(), 'pool label outside --labels'
    yp = yp.to_numpy(int); yt = tr.label.to_numpy(int); folds = tr.fold.to_numpy(int)
    tok, m, hid = load_base(a, torch.float32 if a.cpu and not os.environ.get('TPM_CPU_BF16') else torch.bfloat16)
    model, n_lora = W.add_lora(m, a)
    net = Net(model, hid, a.layers, len(a.labels))
    net.train()   # from_pretrained returns eval mode, where the layers skip checkpointing
    n_ck = W.checkpointed(model) if a.cpu else xla_checkpointing(model)
    net, n_tp = place(net, rt, a.tp)
    if n_tp:
        log('tensor-parallel weights:', n_tp)
    params = [p for p in net.parameters() if p.requires_grad]
    log(f'{rt.n} devices, LoRA params {n_lora}, checkpointed layers {n_ck}, pool {len(pool)}, official {len(tr)}')
    enc_p = W.encode(tok, pool.text.astype(str).tolist(), a.max_len)
    enc_t = W.encode(tok, tr.text.astype(str).tolist(), a.max_len)
    log('tokenised; pool tokens', int(sum(len(x) for x in enc_p)))
    counts = np.bincount(yp, minlength=len(a.labels)).astype(float)
    cw = torch.tensor(len(yp) / (len(a.labels) * np.maximum(counts, 1)), dtype=torch.float32).to(rt.dev)
    rw = pool_weights(pool, a.pool_weight, a.pool_weight_clip) if a.pool_weight > 0 else None
    if rw is not None:
        log(f'pool rows weighted by match_weight ** {a.pool_weight} (clip {a.pool_weight_clip}): effective share '
            f'{float(rw.sum() ** 2 / (rw.astype(np.float64) ** 2).sum() / len(rw)):.3f}')
    rng = np.random.RandomState(a.seed)
    batches = []
    for e in range(int(math.ceil(a.epochs))):
        ep = buckets(W.triplets(pool.prompt_key.astype(str).values, np.arange(len(pool)), rng), enc_p, s1_shapes, rng)
        batches += ep[:int(len(ep) * min(1.0, a.epochs - e))]
    base_meta = dict(base=a.base, keep=a.keep, layers=a.layers, max_len=a.max_len, labels=a.labels, r=a.r, alpha=a.alpha,
                     device='cpu' if a.cpu else f'tpu x{rt.n} spmd', dtype='float32' if a.cpu else 'bfloat16',
                     pool_rows=len(pool), seed=a.seed, pool_weight=a.pool_weight, pool_weight_clip=a.pool_weight_clip)
    log(f'stage 1: {len(batches)} batches over {a.epochs} epochs, shapes {s1_shapes}')
    r1 = train_loop(net, params, batches, enc_p, yp, cw, tok.pad_token_id, rt, a.lr, a.head_lr, a.warmup, a.hours, 'stage 1', rw)
    s1_meta = dict(base_meta, stage=1, lr=a.lr, head_lr=a.head_lr, epochs=a.epochs, **r1,
                   epochs_done=round(a.epochs * r1['steps'] / max(1, r1['planned']), 3))
    s1_dir = a.out.rstrip('/') + '_s1'
    save(model, net, s1_dir, s1_meta)
    lp1 = predict(net, enc_t, list(range(len(tr))), tok.pad_token_id, rt, ev_shapes)
    np.save(os.path.join(os.path.dirname(a.out.rstrip('/')) or '.', 's1_on_train.npy'), lp1)
    s1_meta['official_balanced_accuracy'] = bacc(lp1, yt)
    json.dump(s1_meta, open(os.path.join(s1_dir, 'meta.json'), 'w'), indent=1)
    log(f"stage 1 done {r1}; pool-only model on the official rows: balanced accuracy {s1_meta['official_balanced_accuracy']:.4f}")
    keep = {n: p.detach().cpu().clone() for n, p in net.named_parameters() if p.requires_grad}
    oof = np.zeros((len(tr), len(a.labels)), np.float32); seen = np.zeros(len(tr), bool)
    summary = dict(stage1=s1_meta, jobs={})
    groups = tr.prompt_id.astype(str).values
    for job in a.jobs.split(','):
        with torch.no_grad():
            for n, p in net.named_parameters():
                if n in keep:
                    p.copy_(keep[n].to(p.device))
        if a.tp > 1 and not a.cpu:   # re-annotate the reset factors
            shard_tp(net.model, rt)
        rows = np.arange(len(tr)) if job == 'full' else np.where(folds != int(job))[0]
        jr = np.random.RandomState(a.seed + (99 if job == 'full' else int(job)))
        b2 = []
        for e in range(int(math.ceil(a.epochs2))):
            ep = buckets(W.triplets(groups, rows, jr), enc_t, s2_shapes, jr)
            b2 += ep[:int(len(ep) * min(1.0, a.epochs2 - e))]
        r2 = train_loop(net, params, b2, enc_t, yt, None, tok.pad_token_id, rt, a.lr2, a.head_lr2, a.warmup2, None, f'job {job}')
        name = os.path.basename(a.out.rstrip('/')) + ('' if job == 'full' else f'_f{job}')
        out = os.path.join(os.path.dirname(a.out.rstrip('/')) or '.', name)
        check = predict(net, enc_t, list(range(40)), tok.pad_token_id, rt, ev_shapes)
        meta = dict(base_meta, stage=2, job=job, lr=a.lr2, head_lr=a.head_lr2, epochs=a.epochs2, **r2,
                    stage1=os.path.basename(s1_dir), in_sample_check_40=float((check.argmax(1) == yt[:40]).mean()))
        if job != 'full':
            v = np.where(folds == int(job))[0]
            lp = predict(net, enc_t, list(v), tok.pad_token_id, rt, ev_shapes)
            oof[v] = lp; seen[v] = True
            meta['valid_balanced_accuracy'] = bacc(lp, yt[v])
        save(model, net, out, meta)
        np.save(os.path.join(out, 'check40.npy'), check)
        summary['jobs'][job] = {k: meta[k] for k in meta if k not in base_meta}
        log(f"job {job} done: {r2}, in-sample 40 {meta['in_sample_check_40']:.2f}"
            + (f", held-out balanced accuracy {meta['valid_balanced_accuracy']:.4f}" if job != 'full' else ''))
    d = os.path.dirname(a.out.rstrip('/')) or '.'
    if seen.all():
        np.savez(os.path.join(d, 'oof.npz'), ids=np.array(tr.id.astype(str).tolist()), labels=np.array(a.labels), logits=oof)
        summary['oof_balanced_accuracy'] = bacc(oof, yt)
        log(f"out-of-fold balanced accuracy {summary['oof_balanced_accuracy']:.4f} (stage 1 alone {s1_meta['official_balanced_accuracy']:.4f})")
    json.dump(summary, open(os.path.join(d, 'summary.json'), 'w'), indent=1)
    log('ALL DONE')


if __name__ == '__main__':
    main()

In [ ]:
# ---- driver cell (inserted by kaggle/member/build_tpu_member.py, which fills CONFIG)
# LoRA24 members trained end to end on the TPU v5e-8 (tpu_member.py), one after another in this session: stage 1 on
# crtproduction's open pool (every label, minus anything regenerated from a train prompt or equal to a train text), then
# stage 2 on the 2,699 official rows once per held-out fold (exp015's split, so the out-of-fold log-probs line up with
# the replica) and once on all. Each member's stage-1 time box is cut to what is left of CONFIG['session_hours'].
import glob, hashlib, json, os, subprocess, sys, time
from pathlib import Path
import pandas as pd

CONFIG = {'members': [{'tag': 'll8op1', 'base_parts': ['/llama-3-1-8b-base/'], 'hours': 2.3, 'after_hours': 0.5, 'args': ['--epochs', '2', '--epochs2', '2', '--pool-weight', '1.0', '--pool-weight-clip', '4']}, {'tag': 'q14op1', 'base_parts': ['/qwen-3/', '/14b-base/'], 'hours': 3.0, 'after_hours': 0.8, 'args': ['--keep', '27', '--tp', '4', '--epochs', '2', '--epochs2', '2', '--pool-weight', '1.0', '--pool-weight-clip', '4']}], 'session_hours': 7.5, 'labels': ['claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max'], 'train_sha256': '40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707', 'folds': {'p000': 0, 'p001': 3, 'p002': 1, 'p003': 0, 'p004': 2, 'p005': 1, 'p006': 3, 'p007': 4, 'p008': 3, 'p009': 0, 'p010': 4, 'p011': 3, 'p012': 2, 'p013': 4, 'p014': 0, 'p015': 3, 'p016': 4, 'p017': 4, 'p018': 1, 'p019': 2, 'p020': 2, 'p021': 4, 'p022': 0, 'p023': 1, 'p024': 2, 'p025': 4, 'p026': 4, 'p027': 2, 'p028': 2, 'p029': 3, 'p030': 1, 'p031': 3, 'p032': 3, 'p033': 2, 'p034': 1, 'p035': 0, 'p036': 3, 'p037': 0, 'p038': 0, 'p039': 3, 'p040': 1, 'p041': 1, 'p042': 1, 'p043': 3, 'p044': 1, 'p045': 3, 'p046': 2, 'p047': 4, 'p048': 3, 'p049': 0, 'p050': 3, 'p051': 1, 'p052': 4, 'p053': 2, 'p054': 3, 'p055': 3, 'p056': 2, 'p057': 0, 'p058': 4, 'p059': 0, 'p060': 0, 'p061': 3, 'p062': 1, 'p063': 4, 'p064': 4, 'p065': 2, 'p066': 1, 'p067': 2, 'p068': 4, 'p069': 0, 'p070': 1, 'p071': 4, 'p072': 2, 'p073': 2, 'p074': 2, 'p075': 4, 'p076': 3, 'p077': 3, 'p078': 1, 'p079': 4, 'p080': 3, 'p081': 0, 'p082': 3, 'p083': 2, 'p084': 1, 'p085': 0, 'p086': 2, 'p087': 2, 'p088': 0, 'p089': 0, 'p090': 2, 'p091': 1, 'p092': 4, 'p093': 3, 'p094': 3, 'p095': 1, 'p096': 1, 'p097': 2, 'p098': 2, 'p099': 2, 'p100': 0, 'p101': 2, 'p102': 3, 'p103': 1, 'p104': 0, 'p105': 4, 'p106': 1, 'p107': 3, 'p108': 4, 'p109': 0, 'p110': 4, 'p111': 0, 'p112': 0, 'p113': 4, 'p114': 0, 'p115': 0, 'p116': 1, 'p117': 3, 'p118': 4, 'p119': 3, 'p120': 4, 'p121': 0, 'p122': 1, 'p123': 4, 'p124': 0, 'p125': 2, 'p126': 4, 'p127': 3, 'p128': 1, 'p129': 3, 'p130': 4, 'p131': 1, 'p132': 4, 'p133': 0, 'p134': 0, 'p135': 1, 'p136': 1, 'p137': 0, 'p138': 4, 'p139': 0, 'p140': 1, 'p141': 2, 'p142': 2, 'p143': 3, 'p144': 2, 'p145': 1, 'p146': 2, 'p147': 4, 'p148': 2, 'p149': 1}, 'max_group': 24}
WORK, TMP = Path("/kaggle/working"), Path("/tmp/tpm")
STARTED = time.time()


def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"$ {cmd}\n{(r.stdout + r.stderr)[-3000:]}", flush=True)
    return r.returncode


PKGS = 'pip list 2>/dev/null | grep -iE "^(torch|torch[-_]xla|libtpu|transformers|peft|accelerate|tokenizers|safetensors) "'
sh(PKGS)
sh('python -c "import torch_xla.runtime as xr; print(xr.device_type(), xr.global_runtime_device_count())"')
if sh('python -c "import peft"'):
    import torch   # never let pip replace the TPU build of torch
    open("/tmp/constraints.txt", "w").write(f"torch=={torch.__version__}\n")
    sh("pip install -q peft -c /tmp/constraints.txt")
    sh(PKGS)
cfgs = sorted(glob.glob("/kaggle/input/**/config.json", recursive=True))
train_csv = sorted(glob.glob("/kaggle/input/**/name-that-llm/train.csv", recursive=True))[0]
assert hashlib.sha256(Path(train_csv).read_bytes()).hexdigest() == CONFIG["train_sha256"], "train.csv changed"
train = pd.read_csv(train_csv, dtype=str, keep_default_na=False)
TMP.mkdir(parents=True, exist_ok=True)
train["label"] = train.model.map({l: i for i, l in enumerate(CONFIG["labels"])}).astype(int)
train["fold"] = train.prompt_id.map(CONFIG["folds"]).astype(int)
train[["id", "text", "label", "fold", "prompt_id"]].to_parquet(TMP / "train.parquet", index=False)
op_path = sorted(glob.glob("/kaggle/input/**/ntl_open_pool.parquet", recursive=True))[0]
op = pd.read_parquet(op_path)
n0 = len(op)
op = op[op.model.isin(CONFIG["labels"]) & ~op.flag_train_prompt_regen.astype(bool)]
op = op[~op.text.astype(str).isin(set(train.text))].rename(columns={"prompt_group": "prompt_key"})
op["prompt_key"] = op.prompt_key.astype(str)
if CONFIG.get("max_group"):   # a whole source file shares one prompt_group: no same-prompt triplets from it
    big = op.prompt_key.map(op.prompt_key.value_counts()) > CONFIG["max_group"]
    op.loc[big, "prompt_key"] = "solo_" + pd.Series(range(int(big.sum())), index=op.index[big]).astype(str)
    print(f"{int(big.sum())} rows in groups of more than {CONFIG['max_group']} rows treated as singletons", flush=True)
op[["text", "model", "prompt_key", "match_weight"]].to_parquet(TMP / "pool.parquet", index=False)
print(f"open pool {op_path}: {n0} rows, {len(op)} kept", op.model.value_counts().to_dict(), flush=True)
results = {}
for mem in CONFIG["members"]:
    base = [os.path.dirname(c) for c in cfgs if all(p in c.lower() for p in mem["base_parts"])]
    assert base, (mem["tag"], cfgs[:20])
    left = CONFIG["session_hours"] - (time.time() - STARTED) / 3600 - mem["after_hours"]
    hours = min(mem["hours"], left)
    if hours < 0.5:
        print(f"SKIP {mem['tag']}: {left:.2f} h left in the session", flush=True)
        continue
    out = WORK / mem["tag"]
    args = [sys.executable, "-u", str(WORK / "tpu_member.py"), "--base", base[0], "--labels", json.dumps(CONFIG["labels"]),
            "--pool", str(TMP / "pool.parquet"), "--train", str(TMP / "train.parquet"), "--out", str(out),
            "--hours", f"{hours:.2f}", *mem["args"]]
    print(f"MEMBER {mem['tag']} on {base[0]}, stage-1 time box {hours:.2f} h", flush=True)
    rc = subprocess.run(args, env=dict(os.environ, PJRT_DEVICE="TPU", TOKENIZERS_PARALLELISM="false")).returncode
    results[mem["tag"]] = rc
    print("EXIT", mem["tag"], rc, f"{(time.time() - STARTED) / 3600:.2f} h", flush=True)
    for f in ("summary.json", "oof.npz", "s1_on_train.npy"):   # tpu_member.py writes these next to the member's dirs
        if (WORK / f).exists():
            (WORK / f).rename(WORK / f"{mem['tag']}_{f}")
    if (WORK / f"{mem['tag']}_summary.json").exists():
        print((WORK / f"{mem['tag']}_summary.json").read_text(), flush=True)
print("RESULTS", results, flush=True)
assert results and all(rc == 0 for rc in results.values()), "a member failed"